In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
"""
ALL-IN-ONE GOOGLE COLAB SCRIPT
Downloads US ZIP codes, filters for Gainesville, and creates the internet coverage map.
No need to upload shapefiles to Google Drive!

Instructions:
1. Upload your CSV files to Google Drive:
   - foreign_born_zip_codes.csv
   - internet_coverage_zip_codes.csv
2. Update the DRIVE_PATH variable (line 33) to match your folder location
3. Run this entire script
4. Download the output HTML map from your Google Drive
"""

import pandas as pd
import folium
import json
import geopandas as gpd
import requests
import zipfile
import os
import shutil
from shapely.geometry import shape
from google.colab import drive
from folium.plugins import FloatImage
import base64
from io import BytesIO
from PIL import Image, ImageDraw

print("="*70)
print("GAINESVILLE INTERNET COVERAGE MAP - ALL-IN-ONE SCRIPT")
print("="*70)

# =========================
# CONFIGURATION
# =========================

# Mount Google Drive (if your CSV files are there)
MOUNT_DRIVE = True  # Set to False if uploading CSVs directly to Colab

if MOUNT_DRIVE:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/capstone/'  # Update to your folder path

    # CSV file paths (from Google Drive)
    FOREIGN_BORN_FILE = DRIVE_PATH + 'foreign_born_zip_codes.csv'
    INTERNET_COVERAGE_FILE = DRIVE_PATH + 'internet_coverage_zip_codes.csv'
    OUTPUT_FILE = DRIVE_PATH + 'gainesville_zipcode_internet_coverage_map.html'
else:
    # CSV file paths (uploaded directly to Colab)
    FOREIGN_BORN_FILE = 'foreign_born_zip_codes.csv'
    INTERNET_COVERAGE_FILE = 'internet_coverage_zip_codes.csv'
    OUTPUT_FILE = 'gainesville_zipcode_internet_coverage_map.html'

# Gainesville ZIP codes (city limits only, not full Alachua County)
GAINESVILLE_ZIPS = [
    '32601', '32603', '32605', '32606', '32607', '32608',
    '32609', '32653', '32641','32653', '32669'
]


# =========================
# STEP 1: DOWNLOAD & FILTER ZIP CODE SHAPEFILE
# =========================

print("\n" + "="*70)
print("STEP 1: DOWNLOADING US ZIP CODE SHAPEFILE")
print("="*70)

# Use cartographic boundary file (smaller, faster)
URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_zcta520_500k.zip"
ZIP_FILENAME = "/tmp/us_zipcodes.zip"
SHP_FILENAME = "cb_2020_us_zcta520_500k.shp"
zip_col_name = "ZCTA5CE20"
FILTERED_SHAPEFILE = "/tmp/gainesville_zipcodes.shp"

print(f"\nDownloading from Census Bureau (~100 MB)...")
print("This will take 1-2 minutes...\n")

response = requests.get(URL, stream=True)
total_size = int(response.headers.get('content-length', 0))
downloaded = 0
block_size = 8192

with open(ZIP_FILENAME, 'wb') as f:
    for chunk in response.iter_content(chunk_size=block_size):
        if chunk:
            f.write(chunk)
            downloaded += len(chunk)
            percent = (downloaded / total_size) * 100 if total_size > 0 else 0
            print(f"\rProgress: {percent:.1f}% ({downloaded / (1024*1024):.1f} MB / {total_size / (1024*1024):.1f} MB)", end='')

print("\n✓ Download complete!\n")

# Extract
print("Extracting shapefile...")
extract_dir = "/tmp/temp_shapefiles"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(ZIP_FILENAME, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("✓ Extraction complete!\n")

# Load and filter
print("Loading US shapefile...")
us_shp_path = f"{extract_dir}/{SHP_FILENAME}"
us_zip_boundaries = gpd.read_file(us_shp_path)
print(f"✓ Loaded {len(us_zip_boundaries):,} US ZIP codes\n")

print(f"Filtering for Gainesville ({len(GAINESVILLE_ZIPS)} ZIPs)...")
alachua_zips_gdf = us_zip_boundaries[
    us_zip_boundaries[zip_col_name].isin(GAINESVILLE_ZIPS)
].copy()
print(f"✓ Found {len(alachua_zips_gdf)} Gainesville ZIP codes\n")

# Save filtered shapefile
alachua_zips_gdf.to_file(FILTERED_SHAPEFILE)

# Convert to GeoJSON for folium
zip_boundaries = json.loads(alachua_zips_gdf.to_json())

# Cleanup
os.remove(ZIP_FILENAME)
shutil.rmtree(extract_dir)
print("✓ Temporary files cleaned up\n")


# =========================
# STEP 2: LOAD CSV DATA
# =========================

print("="*70)
print("STEP 2: LOADING CSV DATA")
print("="*70)

print("\nLoading foreign-born data...")
foreign_born = pd.read_csv(FOREIGN_BORN_FILE)
foreign_born_total = foreign_born[foreign_born.iloc[:, 0].str.strip() == "Total"].copy()

# Extract foreign-born totals by ZIP code
fb_data = []
for col in foreign_born_total.columns[1:]:
    zip_code = str(col).zfill(5)
    total = foreign_born_total[col].values[0]
    fb_data.append({
        'ZIP_CODE': zip_code,
        'Foreign_Born_Total': total
    })
foreign_born_df = pd.DataFrame(fb_data)
print(f"✓ Loaded foreign-born data for {len(foreign_born_df)} ZIP codes\n")

print("Loading internet coverage data...")
internet_coverage = pd.read_csv(INTERNET_COVERAGE_FILE)
internet_coverage.columns = ['ZIP_CODE', 'Coverage_Percentage']
internet_coverage['ZIP_CODE'] = internet_coverage['ZIP_CODE'].astype(str).str.zfill(5)
internet_coverage['Coverage_Percentage'] = pd.to_numeric(internet_coverage['Coverage_Percentage'], errors='coerce')
print(f"✓ Loaded internet coverage for {len(internet_coverage)} ZIP codes\n")


# =========================
# STEP 3: MERGE DATA WITH BOUNDARIES
# =========================

print("="*70)
print("STEP 3: MERGING DATA")
print("="*70)

for feature in zip_boundaries['features']:
    zip_code = feature['properties'].get(zip_col_name)

    if zip_code:
        zip_code = str(zip_code).zfill(5)

        # Add internet coverage
        coverage_row = internet_coverage[internet_coverage['ZIP_CODE'] == zip_code]
        if not coverage_row.empty:
            feature['properties']['Coverage_Percentage'] = float(coverage_row['Coverage_Percentage'].values[0])
        else:
            feature['properties']['Coverage_Percentage'] = None

        # Add foreign-born data
        fb_row = foreign_born_df[foreign_born_df['ZIP_CODE'] == zip_code]
        if not fb_row.empty:
            feature['properties']['Foreign_Born_Total'] = int(fb_row['Foreign_Born_Total'].values[0])
        else:
            feature['properties']['Foreign_Born_Total'] = None

print("✓ Data merged successfully\n")


# =========================
# STEP 4: CREATE MAP
# =========================

print("="*70)
print("STEP 4: CREATING MAP")
print("="*70)

# Center map on Gainesville, FL
center_lat = 29.6516
center_lon = -82.3248

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# Calculate coverage statistics
valid_coverage = [f['properties']['Coverage_Percentage']
                  for f in zip_boundaries['features']
                  if f['properties'].get('Coverage_Percentage') is not None]

# Calculate foreign-born population statistics
foreign_born_values = [f['properties'].get('Foreign_Born_Total', 0)
                      for f in zip_boundaries['features']
                      if f['properties'].get('Foreign_Born_Total') is not None and f['properties'].get('Foreign_Born_Total', 0) > 0]

if foreign_born_values:
    sorted_fb = sorted(foreign_born_values)
    min_foreign_born = sorted_fb[0]
    max_foreign_born = sorted_fb[-1]

    print(f"Foreign-born population range: {min_foreign_born:,} - {max_foreign_born:,}\n")
else:
    min_foreign_born = 0
    max_foreign_born = 1

if valid_coverage:
    min_coverage = min(valid_coverage)
    max_coverage = max(valid_coverage)

    # Add title
    title_html = '''
    <div style="position: fixed;
                top: 10px;
                left: 50%;
                transform: translateX(-50%);
                width: auto;
                background-color: white;
                border: 2px solid grey;
                z-index: 9999;
                font-size: 18px;
                font-weight: bold;
                padding: 12px 20px;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
                text-align: center;
                ">
    Internet Coverage and Foreign-Born Population Density
    </div>
    '''

    # Create custom legend HTML at top left
    legend_html = f'''
    <div style="position: fixed;
                top: 60px;
                left: 10px;
                width: 220px;
                height: auto;
                background-color: white;
                border: 2px solid grey;
                z-index: 9999;
                font-size: 14px;
                padding: 10px;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
                ">
    <p style="margin: 0 0 10px 0; font-weight: bold;">Internet Coverage (%)</p>
    <div style="margin-bottom: 15px;">
        <div style="font-size: 11px; line-height: 1.8;">
            <div><span style="color: #d73027;">■</span> Low (&lt;94%)</div>
            <div><span style="color: #fee08b;">■</span> Medium (94-97%)</div>
            <div><span style="color: #1a9850;">■</span> High (97-99.5%)</div>
            <div><span style="color: #004529;">■</span> Very High (&gt;99.5%)</div>
        </div>
    </div>

    <p style="margin: 15px 0 5px 0; font-weight: bold;">Foreign-Born Population</p>
    <div style="font-size: 11px; line-height: 1.8;">
        <div>Purple diagonal lines</div>
        <div>1 line per 1,000 people</div>
        <div style="margin-top: 5px;">
            <svg width="30" height="20" style="vertical-align: middle; margin-right: 5px;">
                <rect width="30" height="20" fill="lightgray" stroke="black" stroke-width="1"/>
                <line x1="0" y1="0" x2="30" y2="20" stroke="purple" stroke-width="1.2"/>
            </svg>
            <span>~1,000</span>
        </div>
        <div>
            <svg width="30" height="20" style="vertical-align: middle; margin-right: 5px;">
                <rect width="30" height="20" fill="lightgray" stroke="black" stroke-width="1"/>
                <line x1="0" y1="0" x2="30" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="0" y1="10" x2="20" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="10" y1="0" x2="30" y2="10" stroke="purple" stroke-width="1.2"/>
            </svg>
            <span>~3,000</span>
        </div>
        <div>
            <svg width="30" height="20" style="vertical-align: middle; margin-right: 5px;">
                <rect width="30" height="20" fill="lightgray" stroke="black" stroke-width="1"/>
                <line x1="0" y1="0" x2="30" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="0" y1="5" x2="25" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="0" y1="10" x2="20" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="0" y1="15" x2="15" y2="20" stroke="purple" stroke-width="1.2"/>
                <line x1="5" y1="0" x2="30" y2="15" stroke="purple" stroke-width="1.2"/>
                <line x1="10" y1="0" x2="30" y2="10" stroke="purple" stroke-width="1.2"/>
                <line x1="15" y1="0" x2="30" y2="5" stroke="purple" stroke-width="1.2"/>
            </svg>
            <span>~7,000+</span>
        </div>
    </div>
    </div>
    '''

    # Add choropleth layer for internet coverage (without built-in legend)
    choropleth = folium.Choropleth(
        geo_data=zip_boundaries,
        name='Internet Coverage',
        data=internet_coverage,
        columns=['ZIP_CODE', 'Coverage_Percentage'],
        key_on=f'feature.properties.{zip_col_name}',
        fill_color='RdYlGn',
        fill_opacity=0.7,
        line_opacity=0.8,
        line_color='black',
        line_weight=1.5,
        nan_fill_color='lightgray',
        nan_fill_opacity=0.3,
        threshold_scale=[
            min_coverage,
            94.0,
            96.0,
            97.0,
            97.5,
            98.0,
            98.5,
            99.0,
            99.5,
            max_coverage
        ],
        show=True
    )
    choropleth.add_to(m)

    # Remove the default legend from choropleth
    for key in choropleth._children:
        if key.startswith('color_map'):
            del(choropleth._children[key])

    # Add hatching pattern overlay for foreign-born population
    print("Adding hatching pattern overlay for foreign-born population...")
    print(f"Creating patterns: 1 line per 1,000 people...")

    def get_pattern_svg(foreign_born_total, pattern_id):
        """Generate SVG pattern with spacing based on population (1 line per 1000)"""
        if foreign_born_total is None or foreign_born_total == 0:
            return ''

        # Calculate line spacing: more population = closer lines
        # Base spacing for 1000 people
        base_spacing = 20  # pixels between lines for 1000 people

        # Calculate actual spacing inversely proportional to population
        # Higher population = smaller spacing = more lines
        spacing = max(3, base_spacing * 1000 / max(1000, foreign_born_total))
        spacing = min(spacing, 40)  # Cap minimum line density

        # Pattern needs to be at least 2x spacing to show variation
        pattern_size = int(spacing * 2)

        # Create diagonal lines with calculated spacing
        lines_svg = f'<line x1="0" y1="0" x2="{pattern_size}" y2="{pattern_size}" stroke="purple" stroke-width="1.2" opacity="0.75"/>'
        lines_svg += f'<line x1="-{spacing}" y1="{spacing}" x2="{spacing}" y2="{pattern_size + spacing}" stroke="purple" stroke-width="1.2" opacity="0.75"/>'
        lines_svg += f'<line x1="{pattern_size - spacing}" y1="-{spacing}" x2="{pattern_size + spacing}" y2="{spacing}" stroke="purple" stroke-width="1.2" opacity="0.75"/>'

        return f'''
        <pattern id="{pattern_id}" patternUnits="userSpaceOnUse" width="{pattern_size}" height="{pattern_size}">
            {lines_svg}
        </pattern>
        '''

    # Create SVG overlay layer
    svg_patterns = ""
    pattern_counter = 0

    # Track patterns for debugging
    pattern_info = []

    for feature in zip_boundaries['features']:
        foreign_born_total = feature['properties'].get('Foreign_Born_Total', 0)
        zip_code = feature['properties'].get(zip_col_name)
        coverage_pct = feature['properties'].get('Coverage_Percentage')

        if foreign_born_total and foreign_born_total > 0:
            pattern_id = f"pattern_{pattern_counter}"
            pattern_counter += 1

            # Get pattern SVG
            pattern_svg = get_pattern_svg(foreign_born_total, pattern_id)
            svg_patterns += pattern_svg

            pattern_info.append(f"ZIP {zip_code}: {foreign_born_total:,} people")

            # Add GeoJson with pattern fill
            style_function = lambda x, pid=pattern_id: {
                'fillColor': f'url(#{pid})',
                'fillOpacity': 1,
                'color': 'transparent',
                'weight': 0
            }

            folium.GeoJson(
                feature,
                style_function=style_function,
                tooltip=folium.GeoJsonTooltip(
                    fields=[zip_col_name, 'Coverage_Percentage', 'Foreign_Born_Total'],
                    aliases=['ZIP Code:', 'Internet Coverage (%):', 'Foreign Born Total:'],
                    sticky=True,
                    labels=True
                ),
            ).add_to(m)

    # Add SVG definitions to map
    if svg_patterns:
        svg_defs = f'''
        <svg style="position: absolute; width: 0; height: 0;">
            <defs>
                {svg_patterns}
            </defs>
        </svg>
        '''
        m.get_root().html.add_child(folium.Element(svg_defs))

    # Add the custom legend and title
    m.get_root().html.add_child(folium.Element(title_html))
    m.get_root().html.add_child(folium.Element(legend_html))

    # Add compass rose
    compass_html = '''
    <div style="position: fixed;
                bottom: 30px;
                right: 30px;
                z-index: 9999;">
        <svg width="80" height="80" viewBox="0 0 100 100">
            <!-- Outer circle -->
            <circle cx="50" cy="50" r="45" fill="white" stroke="black" stroke-width="2" opacity="0.9"/>

            <!-- North arrow (red) -->
            <polygon points="50,10 45,50 50,45 55,50" fill="red" stroke="black" stroke-width="1"/>

            <!-- South arrow (white) -->
            <polygon points="50,90 55,50 50,55 45,50" fill="white" stroke="black" stroke-width="1"/>

            <!-- East arrow -->
            <polygon points="90,50 50,55 55,50 50,45" fill="white" stroke="black" stroke-width="1"/>

            <!-- West arrow -->
            <polygon points="10,50 50,45 45,50 50,55" fill="white" stroke="black" stroke-width="1"/>

            <!-- Center circle -->
            <circle cx="50" cy="50" r="5" fill="black"/>

            <!-- Cardinal directions text -->
            <text x="50" y="8" text-anchor="middle" font-size="12" font-weight="bold" fill="black">N</text>
            <text x="50" y="98" text-anchor="middle" font-size="10" fill="black">S</text>
            <text x="95" y="55" text-anchor="middle" font-size="10" fill="black">E</text>
            <text x="5" y="55" text-anchor="middle" font-size="10" fill="black">W</text>
        </svg>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(compass_html))

    print(f"✓ Hatching pattern overlay added (1 line per 1,000 people)\n")
    print(f"✓ Created {pattern_counter} unique patterns\n")
    for info in pattern_info:
        print(f"   {info}")
    print(f"\n✓ Title, legend, and compass added\n")

print("✓ Map created successfully\n")


# =========================
# STEP 5: SAVE MAP
# =========================

print("="*70)
print("STEP 5: SAVING MAP")
print("="*70)

m.save(OUTPUT_FILE)

print("\n" + "="*70)
print("SUCCESS!")
print("="*70)
print(f"\n✓ Map saved to: {OUTPUT_FILE}")
if valid_coverage:
    print(f"✓ Internet coverage range: {min_coverage:.1f}% - {max_coverage:.1f}%")
if foreign_born_values:
    print(f"✓ Foreign-born population range: {min_foreign_born:,} - {max_foreign_born:,}")
    print(f"✓ Hatching patterns: Variable spacing (1 line per 1,000 people)")
print(f"✓ Number of ZIP codes: {len(zip_boundaries['features'])}")
print("\n" + "="*70)

GAINESVILLE INTERNET COVERAGE MAP - ALL-IN-ONE SCRIPT

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STEP 1: DOWNLOADING US ZIP CODE SHAPEFILE

This will take 1-2 minutes...

Progress: 100.0% (63.6 MB / 63.6 MB)
✓ Download complete!

Extracting shapefile...
✓ Extraction complete!

Loading US shapefile...
✓ Loaded 33,791 US ZIP codes

Filtering for Gainesville (17 ZIPs)...
✓ Found 11 Gainesville ZIP codes

✓ Temporary files cleaned up

STEP 2: LOADING CSV DATA

Loading foreign-born data...
✓ Loaded foreign-born data for 10 ZIP codes

Loading internet coverage data...
✓ Loaded internet coverage for 10 ZIP codes

STEP 3: MERGING DATA
✓ Data merged successfully

STEP 4: CREATING MAP
Foreign-born population range: 592 - 9,950

Adding hatching pattern overlay for foreign-born population...
Creating patterns: 1 line per 1,000 people...
✓ Hatching pattern overlay added (1 line per 1,000